In [1]:
"""
02 - Build training dataset: match NLP flavor sentiment to FlavorGraph embeddings.

Outputs:
- data/training_data.pkl: dict with 'X' (n_samples, 300), 'y' (n_samples,), 'names'
- data/training_data_summary.csv: human-readable summary
"""
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("__file__" in globals() and __file__ or "notebooks/dummy.py").resolve().parents[1] if "__file__" in globals() else Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

In [3]:
# Load assets
with open(DATA / "flavorgraph_embeddings.pickle", "rb") as f:
    emb = pickle.load(f)
target_map = pd.read_csv(DATA / "mobai_target_nodes.csv")
nlp = pd.read_csv(DATA / "nlp_flavor_sentiment.csv")
print(f"NLP flavors: {len(nlp)}")
print(f"Target map entries: {len(target_map)}")

NLP flavors: 13
Target map entries: 25


/tmp/ipykernel_136173/1832826409.py:3: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  emb = pickle.load(f)


In [4]:
def get_vector(node_or_compound: str) -> np.ndarray | None:
    """Resolve a target spec (possibly compound like 'black_tea+milk') to vector."""
    if not isinstance(node_or_compound, str) or not node_or_compound:
        return None
    if "+" in node_or_compound:
        # Compound proxy - average vectors
        parts = node_or_compound.split("+")
        vecs = []
        for p in parts:
            # Look up the part name -> node_id from target_map flavorgraph_node column
            match = target_map[target_map["flavorgraph_node"] == p]
            if not match.empty:
                nid = str(match.iloc[0]["node_id"])
                if nid in emb:
                    vecs.append(emb[nid])
        if vecs:
            return np.mean(vecs, axis=0)
        return None
    else:
        match = target_map[target_map["flavorgraph_node"] == node_or_compound]
        if not match.empty:
            nid = str(match.iloc[0]["node_id"])
            if nid in emb:
                return emb[nid]
        return None

In [5]:
# Build training samples
samples = []
for _, nlp_row in nlp.iterrows():
    flavor = nlp_row["flavor_name"]
    target = target_map[target_map["mobai_term"] == flavor]
    if target.empty:
        print(f"  SKIP {flavor}: no target mapping")
        continue
    fg_node = target.iloc[0]["flavorgraph_node"]
    vec = get_vector(fg_node)
    if vec is None:
        print(f"  SKIP {flavor}: no vector for '{fg_node}'")
        continue
    samples.append({
        "flavor_name": flavor,
        "flavorgraph_node": fg_node,
        "vector": vec,
        "net_sentiment": nlp_row["net_sentiment_pct"],
        "mention_count": nlp_row["mention_count"],
    })
    print(f"  OK   {flavor:12s} -> {fg_node:25s} sentiment={nlp_row['net_sentiment_pct']:+d}")

  OK   mango        -> fresh_mango               sentiment=+53
  OK   vanilla      -> vanilla                   sentiment=+48
  OK   caramel      -> caramel                   sentiment=+44
  OK   coconut      -> coconut                   sentiment=+42
  OK   chocolate    -> chocolate                 sentiment=+41
  OK   milk_tea     -> black_tea+milk            sentiment=+41
  OK   oat          -> oat                       sentiment=+35
  OK   coffee       -> coffee                    sentiment=+34
  OK   banana       -> banana                    sentiment=+28
  OK   matcha       -> matcha_green_tea_powder   sentiment=+28
  OK   peach        -> canned_peach              sentiment=+27
  OK   honey        -> honey                     sentiment=+22
  OK   strawberry   -> strawberry                sentiment=+6


In [6]:
if not samples:
    raise SystemExit("No training samples!")

In [7]:
# Stack into arrays
X = np.stack([s["vector"] for s in samples])
y = np.array([s["net_sentiment"] for s in samples])
names = [s["flavor_name"] for s in samples]

In [8]:
print(f"\n=== Training data ===")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"y range: [{y.min()}, {y.max()}], mean={y.mean():.1f}, std={y.std():.1f}")


=== Training data ===
X shape: (13, 300)
y shape: (13,)
y range: [6, 53], mean=34.5, std=12.0


In [9]:
# Save
out = {"X": X, "y": y, "names": names, "feature_dim": X.shape[1]}
with open(DATA / "training_data.pkl", "wb") as f:
    pickle.dump(out, f)

In [10]:
summary = pd.DataFrame([{
    "flavor_name": s["flavor_name"],
    "flavorgraph_node": s["flavorgraph_node"],
    "net_sentiment": s["net_sentiment"],
    "mention_count": s["mention_count"],
    "vector_norm": float(np.linalg.norm(s["vector"])),
} for s in samples])
summary.to_csv(DATA / "training_data_summary.csv", index=False)
print(f"\nSaved: training_data.pkl ({len(samples)} samples), training_data_summary.csv")
print(summary.to_string(index=False))


Saved: training_data.pkl (13 samples), training_data_summary.csv
flavor_name        flavorgraph_node  net_sentiment  mention_count  vector_norm
      mango             fresh_mango             53             73     2.952990
    vanilla                 vanilla             48            300     3.445198
    caramel                 caramel             44             27     1.969025
    coconut                 coconut             42            179     3.419209
  chocolate               chocolate             41            361     3.555396
   milk_tea          black_tea+milk             41             37     2.605850
        oat                     oat             35            386     3.280564
     coffee                  coffee             34            542     3.205220
     banana                  banana             28            281     3.301369
     matcha matcha_green_tea_powder             28            207     1.744639
      peach            canned_peach             27             26